# LeetCode 2187 — Minimum Time to Complete Trips bằng Python

Link đề: https://leetcode.com/problems/minimum-time-to-complete-trips/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search

Skeleton LeetCode:

```python
class Solution:
    def minimumTime(self, time: List[int], totalTrips: int) -> int:
        ...
```

Bài kinh điển **tìm nhị phân trên đáp án**: hàm "số chuyến xong trong thời gian `t`" `f(t) = Σ ⌊t / time[i]⌋` **không giảm** theo `t`, cần `t` nhỏ nhất với `f(t) ≥ totalTrips`. Notebook đi từ **mô phỏng từng đơn vị thời gian** → **mô phỏng sự kiện bằng heap** → **tìm nhị phân trên `t`** → **tìm nhị phân với cận chặt + gom thời gian trùng** → **`bisect` với `key`** của Python 3.10.

## 1. Mục tiêu học

1. Nhận diện mẫu "**tìm `t` nhỏ nhất sao cho điều kiện đơn điệu đúng**" ⇒ tìm nhị phân trên đáp án.
2. Viết hàm kiểm tra `f(t) = Σ ⌊t / time[i]⌋` và chứng minh nó không giảm.
3. Chọn **cận trên** đúng: `min(time) · totalTrips` (xe nhanh nhất một mình cũng đủ).
4. Viết vòng tìm nhị phân "**chặn dưới đầu tiên**" (`lo < hi`, `hi = mid`, `lo = mid + 1`) không lỗi lệch một.
5. So sánh với mô phỏng sự kiện bằng heap — đúng nhưng `O(totalTrips log n)`.

## 2. Đề bài & ví dụ

`time[i]` là thời gian xe buýt `i` chạy **một chuyến**. Mỗi xe chạy liên tục nhiều chuyến (chuyến sau bắt đầu ngay khi chuyến trước xong), các xe độc lập. Trả về thời gian **nhỏ nhất** để **tổng** số chuyến hoàn thành của mọi xe đạt **ít nhất** `totalTrips`.

**Ví dụ 1:** `time = [1,2,3]`, `totalTrips = 5` → `3` (tại `t = 3`: `3 + 1 + 1 = 5` chuyến).

**Ví dụ 2:** `time = [2]`, `totalTrips = 1` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, `1 ≤ time[i], totalTrips ≤ 10⁷`. Đáp án có thể tới `10¹⁴`.

In [1]:
import heapq
import random
import time as _time
from bisect import bisect_left
from collections import Counter
from typing import List

examples = [([1, 2, 3], 5, 3), ([2], 1, 2)]

def trips_at(time, t):
    return sum(t // x for x in time)

for tm, total, exp in examples:
    print(f"time={tm}, totalTrips={total}: " + ", ".join(f"f({t})={trips_at(tm, t)}" for t in range(1, 5)), f"| kỳ vọng {exp}")

time=[1, 2, 3], totalTrips=5: f(1)=1, f(2)=3, f(3)=5, f(4)=7 | kỳ vọng 3
time=[2], totalTrips=1: f(1)=0, f(2)=1, f(3)=1, f(4)=2 | kỳ vọng 2


## 3. Kiến thức nền

### 3.1 Phép chia nguyên `//` đếm số chuyến

Xe có chu kỳ `x` hoàn thành chuyến thứ `j` tại thời điểm `j · x`; tới thời điểm `t` đã xong `⌊t / x⌋` chuyến.

In [2]:
x = 3
print([(t, t // x) for t in range(0, 10)])

[(0, 0), (1, 0), (2, 0), (3, 1), (4, 1), (5, 1), (6, 2), (7, 2), (8, 2), (9, 3)]


### 3.2 Tìm nhị phân "giá trị đầu tiên thỏa"

Với vị từ `P(t)` đơn điệu (sai…sai, đúng…đúng), tìm `t` nhỏ nhất có `P(t)` đúng:

```python
lo, hi = L, R          # biết P(R) đúng
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid
    else: lo = mid + 1
return lo
```

In [3]:
P = lambda t: t * t >= 50
lo, hi = 0, 50
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid
    else: lo = mid + 1
print("t nhỏ nhất với t² ≥ 50:", lo)

t nhỏ nhất với t² ≥ 50: 8


### 3.3 `heapq` cho mô phỏng sự kiện

Heap chứa `(thời điểm xong chuyến kế tiếp, chu kỳ)`; lấy sự kiện sớm nhất, đẩy lại sự kiện tiếp theo của xe đó.

### 3.4 `bisect_left` với `key` (Python ≥ 3.10)

`bisect_left(range(lo, hi), target, key=f)` tìm vị trí đầu tiên `f(x) ≥ target` trên một dãy **ảo** (không tạo list) — vì `range` hỗ trợ truy cập theo chỉ số.

In [4]:
import sys
print("Python", sys.version.split()[0], "| bisect_left(range(0, 51), 50, key=lambda t: t*t) =", bisect_left(range(0, 51), 50, key=lambda t: t * t))

Python 3.13.18 | bisect_left(range(0, 51), 50, key=lambda t: t*t) = 8


## 4. Ý tưởng nền tảng / trực giác

Theo dõi tổng số chuyến theo thời gian cho ví dụ 1 (`time = [1, 2, 3]`):

| `t` | xe 1 | xe 2 | xe 3 | tổng |
|---|---|---|---|---|
| 1 | 1 | 0 | 0 | 1 |
| 2 | 2 | 1 | 0 | 3 |
| 3 | 3 | 1 | 1 | **5** ✓ |

Tổng **không giảm** theo `t` (thời gian trôi, chuyến đã xong không mất). ⇒ tồn tại ngưỡng: mọi `t` trước ngưỡng không đủ, mọi `t` từ ngưỡng trở đi đủ ⇒ tìm nhị phân.

**Cận:** `t = 0` không đủ (`totalTrips ≥ 1`). `t = min(time) · totalTrips` chắc chắn đủ (chỉ xe nhanh nhất đã chạy `totalTrips` chuyến).

## 5. Cách 1 — Brute Force / baseline: tăng `t` từng đơn vị

In [5]:
def min_time_linear(time, totalTrips):
    t = 0
    while True:
        t += 1
        if sum(t // x for x in time) >= totalTrips:
            return t

for tm, total, exp in examples:
    print(min_time_linear(tm, total), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(answer · n)` — `answer` tới `10¹⁴`: không khả thi. **Nút thắt:** thử tuần tự một hàm đơn điệu ⇒ tìm nhị phân `O(n log answer)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đơn điệu:** nếu `t₁ ≤ t₂` thì `⌊t₁/x⌋ ≤ ⌊t₂/x⌋` với mọi `x > 0` ⇒ `f(t₁) ≤ f(t₂)`.

**Tìm nhị phân trên `[1, min(time) · totalTrips]`:**
- **nếu** `f(mid) ≥ totalTrips` **thì** đáp án `≤ mid` ⇒ `hi = mid`;
- **ngược lại** đáp án `> mid` ⇒ `lo = mid + 1`.

Mỗi lần kiểm `O(n)`; số lần `≈ log₂(10¹⁴) ≈ 47`.

**Cận dưới chặt hơn (tùy chọn):** trong thời gian `t`, mỗi xe chạy tối đa `⌊t / min(time)⌋` chuyến (không xe nào nhanh hơn xe nhanh nhất), nên `n · ⌊t / min(time)⌋ ≥ totalTrips` là điều kiện **cần** ⇒ `t ≥ min(time) · ⌈totalTrips / n⌉` là cận dưới hợp lệ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng sự kiện bằng heap

In [6]:
def min_time_heap(time, totalTrips):
    h = [(x, x) for x in time]                 # (thời điểm xong chuyến kế, chu kỳ)
    heapq.heapify(h)
    done = 0
    while True:
        t, x = heapq.heappop(h)
        done += 1
        if done == totalTrips:
            return t
        heapq.heappush(h, (t + x, x))

for tm, total, exp in examples:
    print(min_time_heap(tm, total), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O((n + totalTrips) log n)` — tới `10⁷ · 17` phép heap: quá chậm trong Python, nhưng là cách mô phỏng "đúng nghĩa đen" rất dễ tin ⇒ dùng làm chuẩn đối chiếu với `totalTrips` vừa phải.

### 7.2 Cách 3 — Tìm nhị phân trên `t`

In [7]:
def min_time_bsearch(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // x for x in time) >= totalTrips:
            hi = mid
        else:
            lo = mid + 1
    return lo

for tm, total, exp in examples:
    print(min_time_bsearch(tm, total), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Độ phức tạp:** `O(n log(min(time) · totalTrips))`.

### 7.3 Cách 4 — Cận chặt + gom thời gian trùng + dừng sớm khi đếm

Ba cải tiến hằng số:
1. Cận dưới `min(time) · ⌈totalTrips / n⌉`, cận trên `min(time) · totalTrips`.
2. Gom các xe có **cùng** chu kỳ: `f(t) = Σ_x cnt[x] · ⌊t/x⌋` — ít phép chia hơn khi nhiều giá trị trùng.
3. Khi đếm, **dừng sớm** ngay khi tổng đã `≥ totalTrips`.

In [8]:
def min_time_tuned(time, totalTrips):
    cnt = sorted(Counter(time).items())        # chu kỳ tăng dần: xe nhanh trước -> dừng sớm nhanh hơn
    fastest = cnt[0][0]
    lo = fastest * (-(-totalTrips // len(time)))
    hi = fastest * totalTrips
    def enough(t):
        s = 0
        for x, c in cnt:
            q = t // x
            if q == 0:
                break                           # các chu kỳ sau còn lớn hơn
            s += c * q
            if s >= totalTrips:
                return True
        return False
    while lo < hi:
        mid = (lo + hi) // 2
        if enough(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for tm, total, exp in examples:
    print(min_time_tuned(tm, total), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Lưu ý thực nghiệm (mục 11):** với dữ liệu ngẫu nhiên ít trùng, chi phí `Counter` + sắp xếp làm cách này **chậm hơn** cách 3; nó chỉ thắng lớn khi có nhiều chu kỳ trùng. Tối ưu hằng số phải đo, không đoán.

**Vì sao `break` khi `q == 0` là đúng?** `cnt` sắp chu kỳ tăng dần; nếu `t < x` thì mọi chu kỳ sau cũng `> t` ⇒ đóng góp 0. **Độ phức tạp:** như cách 3 nhưng hằng số nhỏ hơn nhiều với dữ liệu trùng lặp.

### 7.4 Cách 5 — `bisect_left` với `key`

In [9]:
def min_time_bisect_key(time, totalTrips):
    hi = min(time) * totalTrips
    return bisect_left(range(hi + 1), totalTrips, key=lambda t: sum(t // x for x in time))

for tm, total, exp in examples:
    print(min_time_bisect_key(tm, total), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
2 (kỳ vọng 2 )


**Lưu ý:** `range(hi + 1)` không tạo list (O(1) bộ nhớ); `bisect_left` gọi `key` trên `O(log hi)` phần tử. Cần Python ≥ 3.10 (LeetCode hiện dùng bản mới).

## 8. TRACE từng bước

### 8.1 Trace tìm nhị phân trên ví dụ 1

In [10]:
def trace_bsearch(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips; step = 0
    print(f"  time={time}, totalTrips={totalTrips}, khoảng ban đầu [{lo}, {hi}]")
    while lo < hi:
        step += 1
        mid = (lo + hi) // 2; f = sum(mid // x for x in time)
        ok = f >= totalTrips
        print(f"    bước {step}: lo={lo}, hi={hi}, mid={mid}, f(mid)={[mid // x for x in time]} -> {f} {'≥' if ok else '<'} {totalTrips} -> {'hi = mid' if ok else 'lo = mid + 1'}")
        if ok: hi = mid
        else: lo = mid + 1
    print(f"  => {lo}")
    return lo

_ = trace_bsearch([1, 2, 3], 5)

  time=[1, 2, 3], totalTrips=5, khoảng ban đầu [1, 5]
    bước 1: lo=1, hi=5, mid=3, f(mid)=[3, 1, 1] -> 5 ≥ 5 -> hi = mid
    bước 2: lo=1, hi=3, mid=2, f(mid)=[2, 1, 0] -> 3 < 5 -> lo = mid + 1
  => 3


### 8.2 Trace mô phỏng heap trên `time = [3, 5]`, `totalTrips = 6` (ví dụ tự dựng — đã đối chiếu tìm nhị phân, đối chiếu lại đề gốc)

In [11]:
def trace_heap(time, totalTrips):
    h = [(x, x) for x in time]; heapq.heapify(h); done = 0
    while True:
        t, x = heapq.heappop(h); done += 1
        print(f"    t={t:>2}: xe chu kỳ {x} xong một chuyến -> tổng {done}")
        if done == totalTrips:
            return t
        heapq.heappush(h, (t + x, x))

r = trace_heap([3, 5], 6)
print("  =>", r, "| tìm nhị phân:", min_time_bsearch([3, 5], 6))

    t= 3: xe chu kỳ 3 xong một chuyến -> tổng 1
    t= 5: xe chu kỳ 5 xong một chuyến -> tổng 2
    t= 6: xe chu kỳ 3 xong một chuyến -> tổng 3
    t= 9: xe chu kỳ 3 xong một chuyến -> tổng 4
    t=10: xe chu kỳ 5 xong một chuyến -> tổng 5
    t=12: xe chu kỳ 3 xong một chuyến -> tổng 6
  => 12 | tìm nhị phân: 12


## 9. Code nộp LeetCode

In [12]:
class Solution:
    def minimumTime(self, time: List[int], totalTrips: int) -> int:
        lo, hi = 1, min(time) * totalTrips          # hi chắc chắn đủ
        while lo < hi:
            mid = (lo + hi) // 2
            trips = 0
            for x in time:
                trips += mid // x
                if trips >= totalTrips:
                    break
            if trips >= totalTrips:
                hi = mid                             # mid đủ -> đáp án <= mid
            else:
                lo = mid + 1                         # mid chưa đủ -> đáp án > mid
        return lo

for tm, total, exp in examples:
    assert Solution().minimumTime(tm, total) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Đơn điệu:** `f(t) = Σ ⌊t/x⌋` không giảm (mục 6).
2. **Cận:** `f(min(time) · totalTrips) ≥ ⌊min · total / min⌋ = totalTrips` ⇒ `hi` thỏa; `t = 0` không thỏa ⇒ đáp án thuộc `[1, hi]`.
3. **Bất biến tìm nhị phân:** đáp án (giá trị nhỏ nhất thỏa) luôn trong `[lo, hi]`, và `hi` luôn thỏa. Nếu `mid` thỏa, đáp án `≤ mid` ⇒ `hi = mid` giữ bất biến. Nếu không, mọi `t ≤ mid` không thỏa (đơn điệu) ⇒ `lo = mid + 1`. Mỗi vòng thu hẹp khoảng (vì `mid < hi`), dừng khi `lo = hi` = đáp án.
4. **Dừng sớm** khi đếm không làm sai vì chỉ cần biết `trips ≥ totalTrips` hay không.
5. **Mô phỏng heap** lấy các thời điểm hoàn thành theo thứ tự tăng dần; thời điểm của chuyến thứ `totalTrips` chính là `t` nhỏ nhất có `f(t) ≥ totalTrips`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(time)`, `T = totalTrips`, `A` = đáp án `≤ min(time) · T ≤ 10¹⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. Tăng `t` từng bước | `O(A · n)` | `O(1)` |
| 2. Heap sự kiện | `O((n + T) log n)` | `O(n)` |
| 3. Tìm nhị phân | `O(n log A)` | `O(1)` |
| 4. Cận chặt + gom trùng | `O(V log A)` (V = số chu kỳ khác nhau) | `O(V)` |
| 5. `bisect` + `key` | `O(n log A)` | `O(1)` |

In [13]:
rnd = random.Random(2187)
big_time = [rnd.randint(1, 10**7) for _ in range(10**5)]
dup_time = [rnd.choice([3, 7, 10**7]) for _ in range(10**5)]
for label_, tm, total in [("ngẫu nhiên", big_time, 10**7), ("nhiều trùng", dup_time, 10**7), ("1 xe chậm", [10**7], 10**7)]:
    row = []
    for name, fn in [("bsearch", min_time_bsearch), ("tuned", min_time_tuned), ("bisect_key", min_time_bisect_key), ("Solution", Solution().minimumTime)]:
        t0 = _time.perf_counter(); r = fn(tm, total); row.append(f"{name}={(_time.perf_counter() - t0) * 1000:.0f}ms")
    print(f"{label_:<11} -> {r}: " + ", ".join(row))

ngẫu nhiên  -> 64585681: bsearch=86ms, tuned=325ms, bisect_key=68ms, Solution=98ms
nhiều trùng -> 633: bsearch=84ms, tuned=3ms, bisect_key=75ms, Solution=39ms
1 xe chậm   -> 100000000000000: bsearch=0ms, tuned=0ms, bisect_key=0ms, Solution=0ms


## 12. Bộ test

In [14]:
ALL = {
    "bsearch": min_time_bsearch,
    "tuned": min_time_tuned,
    "bisect_key": min_time_bisect_key,
    "Solution": lambda tm, total: Solution().minimumTime(tm, total),
}

def check(time, totalTrips, exp=None, ref="heap"):
    r = min_time_heap(time, totalTrips) if ref == "heap" else min_time_bsearch(time, totalTrips)
    if ref == "heap" and totalTrips <= 200 and r <= 2000:
        assert r == min_time_linear(time, totalTrips)
    for name, fn in ALL.items():
        got = fn(list(time), totalTrips)
        assert got == r, (name, time, totalTrips, got, r)
    assert trips_at(time, r) >= totalTrips and trips_at(time, r - 1) < totalTrips      # validator: tối thiểu
    if exp is not None:
        assert r == exp
    return r

for tm, total, exp in examples:
    check(tm, total, exp)
print("Ví dụ đề: OK")

check([1], 1, 1); check([1], 10**7, 10**7, ref="bs")
check([10**7], 10**7, 10**14, ref="bs")               # đáp án lớn nhất
check([5, 5, 5], 7, 15)                                # trùng chu kỳ
check([3, 5], 6)
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(1500):
    n = rnd.randint(1, 6)
    tm = [rnd.randint(1, 12) for _ in range(n)]
    check(tm, rnd.randint(1, 60))
print("1500 bộ nhỏ so với mô phỏng heap (và tăng t tuyến tính): OK")

for _ in range(10):
    n = rnd.randint(1, 10**5)
    tm = [rnd.randint(1, 10**7) for _ in range(n)]
    check(tm, rnd.randint(1, 10**7), ref="bs")
print("10 bộ lớn (đối chiếu chéo + validator tối thiểu): OK")

Ví dụ đề: OK
Edge case: OK


1500 bộ nhỏ so với mô phỏng heap (và tăng t tuyến tính): OK


10 bộ lớn (đối chiếu chéo + validator tối thiểu): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cận trên quá nhỏ (`hi = totalTrips`)

In [15]:
def bug_small_hi(time, totalTrips):
    lo, hi = 1, totalTrips                      # SAI: khi xe chậm (time > 1), đáp án có thể > totalTrips
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // x for x in time) >= totalTrips: hi = mid
        else: lo = mid + 1
    return lo

print("time=[2], total=1 -> sai:", bug_small_hi([2], 1), "| đúng:", Solution().minimumTime([2], 1))

time=[2], total=1 -> sai: 1 | đúng: 2


**Vì sao sai:** tìm nhị phân chỉ tìm trong khoảng cho trước; nếu đáp án nằm ngoài, nó trả về `hi` (không thỏa). **Sửa:** `hi = min(time) · totalTrips` (hoặc kiểm `f(hi)` trước). Validator "`f(r) ≥ totalTrips`" bắt được lỗi này ngay.

### Lỗi 2 — Điều kiện `>` thay vì `≥`

In [16]:
def bug_strict(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips + min(time)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // x for x in time) > totalTrips: hi = mid      # SAI: "ít nhất" nghĩa là >=
        else: lo = mid + 1
    return lo

print("ví dụ 1 -> sai:", bug_strict([1, 2, 3], 5), "| đúng:", Solution().minimumTime([1, 2, 3], 5))

ví dụ 1 -> sai: 4 | đúng: 3


**Sửa:** `≥ totalTrips`.

### Lỗi 3 — Cập nhật `hi = mid − 1` khi `mid` thỏa (bỏ mất đáp án)

In [17]:
def bug_hi_minus_one(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid // x for x in time) >= totalTrips: hi = mid - 1   # SAI: mid có thể chính là đáp án
        else: lo = mid + 1
    return lo

print("ví dụ 1 -> sai:", bug_hi_minus_one([1, 2, 3], 5), "| đúng:", Solution().minimumTime([1, 2, 3], 5))

ví dụ 1 -> sai: 2 | đúng: 3


**Vì sao sai:** kiểu vòng `lo < hi` với "đáp án đầu tiên thỏa" phải giữ `mid` khi nó thỏa. Muốn dùng `hi = mid − 1` thì phải lưu `ans = mid` riêng và dùng `lo ≤ hi`. Đừng trộn hai khuôn mẫu.

### Lỗi 4 — Dùng phép chia số thực `mid / x`

In [18]:
def bug_float_div(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(mid / x for x in time) >= totalTrips: hi = mid     # SAI: đếm cả "nửa chuyến"
        else: lo = mid + 1
    return lo

print("ví dụ 1 -> sai:", bug_float_div([1, 2, 3], 5), "| đúng:", Solution().minimumTime([1, 2, 3], 5), "(tình cờ đúng!)")
print("time=[3,3], total=1 -> sai:", bug_float_div([3, 3], 1), "| đúng:", Solution().minimumTime([3, 3], 1))

ví dụ 1 -> sai: 3 | đúng: 3 (tình cờ đúng!)
time=[3,3], total=1 -> sai: 2 | đúng: 3


**Vì sao sai:** chuyến **chưa xong** không được tính. Với hai xe chu kỳ 3, tại `t = 2` mỗi xe mới đi được `2/3` chuyến — cộng lại `1.33 ≥ 1` nên bản lỗi trả `2`, nhưng thực tế chưa xe nào xong chuyến. Ví dụ 1 của đề lại **tình cờ** cho đúng — lý do phải test thêm. **Sửa:** `mid // x`.

### Lỗi 5 — Lấy `hi = max(time) · totalTrips` mà không dừng sớm khi đếm ⇒ chậm (không sai)

In [19]:
def count_divisions(time, totalTrips, hi_mode, early):
    lo = 1
    hi = (min(time) if hi_mode == "min" else max(time)) * totalTrips
    ops = 0
    while lo < hi:
        mid = (lo + hi) // 2; s = 0
        for x in time:
            s += mid // x; ops += 1
            if early and s >= totalTrips: break
        if s >= totalTrips: hi = mid
        else: lo = mid + 1
    return lo, ops

tm = [1] + [10**7] * 99999
print("hi=max, không dừng sớm:", count_divisions(tm, 10**7, "max", False))
print("hi=min, dừng sớm      :", count_divisions(tm, 10**7, "min", True))

hi=max, không dừng sớm: (10000000, 4700000)
hi=min, dừng sớm      : (10000000, 2300000)


**Vì sao là lỗi:** kết quả đúng nhưng nhiều vòng hơn (cận rộng hơn) và mỗi vòng chia `n` lần. Với `n = 10⁵` vẫn qua, nhưng thói quen chọn cận chặt + dừng sớm giúp tránh TLE ở các bài nặng hơn.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Thời gian **nhỏ nhất** để đạt **ít nhất** X" + hàm đếm đơn điệu ⇒ **tìm nhị phân trên đáp án**.
- Cận trên: kịch bản "một tài nguyên tốt nhất làm hết"; cận dưới: kịch bản "mọi tài nguyên đều tốt nhất".
- Khuôn mẫu "đáp án đầu tiên thỏa": `while lo < hi: mid; if ok: hi = mid else lo = mid + 1`.
- Đếm có thể **dừng sớm** khi đã đủ.
- Họ bài giống: Koko Eating Bananas (875), Capacity To Ship (1011), Minimized Maximum of Products (2064), Minimum Time to Repair Cars (2594).

## 15. Biến thể của bài

1. **LeetCode 875 — Koko Eating Bananas:** tìm tốc độ nhỏ nhất, hàm đếm `Σ ⌈pile / k⌉` **giảm** theo `k`.
2. **LeetCode 2594 — Minimum Time to Repair Cars:** thợ hạng `r` sửa `c` xe mất `r·c²` ⇒ trong `t` sửa được `⌊√(t/r)⌋` xe.
3. **Mỗi xe có thời gian khởi động** `start[i]`: `f(t) = Σ max(0, ⌊(t − start[i]) / time[i]⌋ + …)` — vẫn đơn điệu.
4. **Tìm số chuyến tối đa trong thời gian `t`** — chính là `f(t)`, không cần tìm nhị phân.
5. **Mỗi xe tối đa `k` chuyến:** `f(t) = Σ min(k, ⌊t/x⌋)` — vẫn đơn điệu, cận trên phải kiểm khả thi (`n·k ≥ totalTrips`).

In [20]:
from math import isqrt

def repair_cars_2594(ranks, cars):
    lo, hi = 1, min(ranks) * cars * cars
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(isqrt(mid // r) for r in ranks) >= cars: hi = mid
        else: lo = mid + 1
    return lo

def koko_875(piles, h):
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(-(-p // mid) for p in piles) <= h: hi = mid
        else: lo = mid + 1
    return lo

def repair_brute(ranks, cars):
    t = 0
    while sum(isqrt(t // r) for r in ranks) < cars:
        t += 1
    return t

rnd_v = random.Random(15)
for _ in range(200):
    rk = [rnd_v.randint(1, 5) for _ in range(rnd_v.randint(1, 4))]; c = rnd_v.randint(1, 8)
    assert repair_cars_2594(rk, c) == repair_brute(rk, c)
print("2594 ranks=[4,2,3,1], cars=10 ->", repair_cars_2594([4, 2, 3, 1], 10), "| 875 piles=[3,6,7,11], h=8 ->", koko_875([3, 6, 7, 11], 8))

2594 ranks=[4,2,3,1], cars=10 -> 16 | 875 piles=[3,6,7,11], h=8 -> 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tăng `t` | thử tuần tự | `O(A n)` | `O(1)` | kiểm thử nhỏ |
| 2. Heap sự kiện | mô phỏng | `O((n+T) log n)` | `O(n)` | đối chiếu, `T` nhỏ |
| 3. Tìm nhị phân | hàm đơn điệu | `O(n log A)` | `O(1)` | **nộp bài** |
| 4. Cận chặt + gom | hằng số nhỏ | `O(V log A)` | `O(V)` | dữ liệu trùng nhiều |
| 5. `bisect` + key | thư viện | `O(n log A)` | `O(1)` | Python 3.10+ gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minimumTime([5, 10, 10], 9)`?

**Bài 2 (tính tay):** Với `time = [2, 3]`, `totalTrips = 7`, lập bảng `f(t)` cho `t = 1..10` và chỉ ra đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `min_time_with_cap(time, totalTrips, k)` — mỗi xe chạy **tối đa `k`** chuyến; trả `-1` nếu không thể.

**Bài 4 (tự viết lại):** Viết lại tìm nhị phân theo khuôn mẫu `lo ≤ hi` với biến `ans`.

**Bài 5 (mini challenge):** Mỗi xe `i` có thời gian **khởi động** `start[i]` trước chuyến đầu (chuyến thứ `j` xong tại `start[i] + j · time[i]`). Tìm thời gian nhỏ nhất đạt `totalTrips`. Kiểm bằng mô phỏng heap.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def min_time_with_cap(time, totalTrips, k):
    # TODO: nếu len(time) * k < totalTrips: -1; f(t) = Σ min(k, t // x)
    pass

# Bài 4 — TODO
def min_time_lo_le_hi(time, totalTrips):
    # TODO: while lo <= hi: if ok: ans = mid; hi = mid - 1 else: lo = mid + 1
    pass

# Bài 5 — TODO
def min_time_with_start(time, start, totalTrips):
    # TODO: f(t) = Σ max(0, (t - start[i]) // time[i]); hi = min(start[i] + time[i] * totalTrips)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `f(t) = ⌊t/5⌋ + 2⌊t/10⌋`.
- Mức 2: `f(25) = 5 + 4 = 9`; `f(24) = 4 + 4 = 8`.
- Mức 3: `25`.

**Bài 2**
- Mức 1: `f(t) = ⌊t/2⌋ + ⌊t/3⌋`.
- Mức 2: `f(8) = 4 + 2 = 6`, `f(9) = 4 + 3 = 7`.
- Mức 3: `9`.

**Bài 3**
- Mức 1: Khả thi ⇔ `n·k ≥ totalTrips`.
- Mức 2: `hi = max(time) · k` (khi đó mọi xe xong `k` chuyến).
- Mức 3: Kiểm bằng heap có giới hạn chuyến mỗi xe.

**Bài 4**
- Mức 1: `ans = hi` ban đầu.
- Mức 2: `mid` thỏa ⇒ `ans = mid`, `hi = mid − 1`.
- Mức 3: Trả `ans`.

**Bài 5**
- Mức 1: Chuyến thứ `j` của xe `i` xong tại `start[i] + j·time[i]` (`j ≥ 1`).
- Mức 2: `f(t) = Σ max(0, ⌊(t − start[i]) / time[i]⌋)`.
- Mức 3: Cận trên: `min_i(start[i] + time[i]·totalTrips)`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
r1 = Solution().minimumTime([5, 10, 10], 9)
print("Bài 1:", r1); assert r1 == 25 == min_time_heap([5, 10, 10], 9)

# Bài 2
print("Bài 2:", [(t, trips_at([2, 3], t)) for t in range(1, 11)], "-> đáp án", Solution().minimumTime([2, 3], 7))
assert Solution().minimumTime([2, 3], 7) == 9

# Bài 3
def min_time_with_cap_sol(time, totalTrips, k):
    if len(time) * k < totalTrips:
        return -1
    lo, hi = 1, max(time) * k
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(min(k, mid // x) for x in time) >= totalTrips: hi = mid
        else: lo = mid + 1
    return lo

def cap_heap(time, totalTrips, k):
    if len(time) * k < totalTrips: return -1
    h = [(x, x, 1) for x in time]; heapq.heapify(h); done = 0
    while True:
        t, x, j = heapq.heappop(h); done += 1
        if done == totalTrips: return t
        if j < k: heapq.heappush(h, (t + x, x, j + 1))

rnd = random.Random(3)
for _ in range(1000):
    tm = [rnd.randint(1, 9) for _ in range(rnd.randint(1, 5))]; T = rnd.randint(1, 30); k = rnd.randint(1, 8)
    assert min_time_with_cap_sol(tm, T, k) == cap_heap(tm, T, k)
print("Bài 3 OK, ví dụ:", min_time_with_cap_sol([1, 2, 3], 5, 2), min_time_with_cap_sol([1, 2], 5, 2))

Bài 1: 25
Bài 2: [(1, 0), (2, 1), (3, 2), (4, 3), (5, 3), (6, 5), (7, 5), (8, 6), (9, 7), (10, 8)] -> đáp án 9
Bài 3 OK, ví dụ: 4 -1


In [23]:
# Bài 4
def min_time_lo_le_hi_sol(time, totalTrips):
    lo, hi = 1, min(time) * totalTrips; ans = hi
    while lo <= hi:
        mid = (lo + hi) // 2
        if sum(mid // x for x in time) >= totalTrips:
            ans = mid; hi = mid - 1
        else:
            lo = mid + 1
    return ans

# Bài 5
def min_time_with_start_sol(time, start, totalTrips):
    lo, hi = 1, min(s + x * totalTrips for x, s in zip(time, start))
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(max(0, (mid - s) // x) for x, s in zip(time, start)) >= totalTrips: hi = mid
        else: lo = mid + 1
    return lo

def start_heap(time, start, totalTrips):
    h = [(s + x, x) for x, s in zip(time, start)]; heapq.heapify(h); done = 0
    while True:
        t, x = heapq.heappop(h); done += 1
        if done == totalTrips: return t
        heapq.heappush(h, (t + x, x))

rnd = random.Random(5)
for _ in range(1000):
    tm = [rnd.randint(1, 9) for _ in range(rnd.randint(1, 5))]; T = rnd.randint(1, 40)
    st = [rnd.randint(0, 10) for _ in tm]
    assert min_time_lo_le_hi_sol(tm, T) == Solution().minimumTime(tm, T)
    assert min_time_with_start_sol(tm, st, T) == start_heap(tm, st, T)
print("Bài 4, Bài 5 OK; ví dụ:", min_time_with_start_sol([1, 2, 3], [5, 0, 0], 5))

Bài 4, Bài 5 OK; ví dụ: 6


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tìm nhị phân trên đáp án:**

```python
lo, hi = 1, min(time) * totalTrips
while lo < hi:
    mid = (lo + hi) // 2
    if sum(mid // x for x in time) >= totalTrips: hi = mid
    else: lo = mid + 1
return lo
```

**Checklist:**
- [ ] `hi` **chắc chắn thỏa** (`min(time) · totalTrips`)?
- [ ] Điều kiện `≥ totalTrips` (không phải `>`)?
- [ ] Khuôn `lo < hi` ⇒ `hi = mid` (không `mid − 1`)?
- [ ] Phép chia nguyên `//`?
- [ ] (Tùy chọn) dừng sớm khi đếm đủ?
- [ ] Chữ ký đúng: `def minimumTime(self, time: List[int], totalTrips: int) -> int`?